<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/Samburu%20GPS%20Plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, json, math
import urllib.request
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from shapely.geometry import shape, Point
from shapely.ops import unary_union

# ================= CONFIG =================
XLSX_PATH    = "Inventory12.xlsx"
CACHE_FILE   = "kenya_wards.geojson"
SAMBURU_FILE = "samburu_wards.geojson"
BOUNDARY_URL = ("https://raw.githubusercontent.com/benaboki/"
                "Kenya-County-Assembly-Boundaries/master/"
                "kenya_county_assemblies.geojson")

COUNTY_COL = "1.County:"
CHAMPION_COL = "a. Graduation Champion Name:"
WARD_COL_SRC = "3.Ward:"
SUBCOUNTY_COL = "2.Sub county:"
LAT_COL = "_8.GPS _latitude"
LON_COL = "_8.GPS _longitude"
HH_HEAD_COL = "5.Name of the Household Head:"

# ================= 1. LOAD & CLEAN =================
df = pd.read_excel(XLSX_PATH)
df.columns = [str(c).strip() for c in df.columns]

df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")
df = df.dropna(subset=[LAT_COL, LON_COL])

df[COUNTY_COL] = df[COUNTY_COL].astype(str).str.strip().str.lower()
df = df[df[COUNTY_COL] == "samburu"].copy()

# Clean champion names: strip whitespace + normalise case for grouping,
# but keep a tidy display version (title case).
df[CHAMPION_COL] = df[CHAMPION_COL].astype(str).str.strip()
df["champion_clean"] = df[CHAMPION_COL].str.lower().str.strip()
# Map back to a single canonical display name per cleaned key (most common raw form)
canonical = (df.groupby("champion_clean")[CHAMPION_COL]
             .agg(lambda s: s.value_counts().idxmax()))
df["champion_display"] = df["champion_clean"].map(canonical)

df[SUBCOUNTY_COL] = df[SUBCOUNTY_COL].astype(str).str.strip()

print(f"{len(df)} GPS rows loaded for Samburu County.")
print(f"{df['champion_display'].nunique()} unique Graduation Champions (after name cleaning).")

# ================= 2. LOAD SAMBURU WARD BOUNDARIES =================
def load_samburu_wards():
    if os.path.exists(SAMBURU_FILE):
        return json.load(open(SAMBURU_FILE))
    if not os.path.exists(CACHE_FILE):
        print("Downloading Kenya ward boundaries (one-time download)...")
        urllib.request.urlretrieve(BOUNDARY_URL, CACHE_FILE)
    national = json.load(open(CACHE_FILE))
    feats = [f for f in national["features"]
             if "samburu" in str(f["properties"].get("county", "")).lower()]
    samburu = {"type": "FeatureCollection", "features": feats}
    json.dump(samburu, open(SAMBURU_FILE, "w"))
    return samburu

wards_gj = load_samburu_wards()
print(f"Wards found in Samburu County boundary file: {len(wards_gj['features'])}")

ward_shapes = [(f["properties"]["ward"],
                f["properties"].get("const", ""),
                shape(f["geometry"])) for f in wards_gj["features"]]

def assign_ward(lon, lat):
    pt = Point(lon, lat)
    for ward, const, geom in ward_shapes:
        if geom.contains(pt):
            return ward, const
    return "Outside mapped wards", ""

wards, consts = zip(*[assign_ward(x, y) for x, y in zip(df[LON_COL], df[LAT_COL])])
df["ward_boundary"] = list(wards)
df["constituency"] = list(consts)

n_outside = (df["ward_boundary"] == "Outside mapped wards").sum()
print(f"{len(df) - n_outside} points matched to a mapped ward boundary; "
      f"{n_outside} fell outside the boundary file (kept, using recorded ward instead).")

# Use the boundary-matched ward where available, otherwise fall back to the
# ward recorded on the household form itself, so no point is silently dropped.
df["ward_final"] = df["ward_boundary"].where(
    df["ward_boundary"] != "Outside mapped wards", df[WARD_COL_SRC]
)

# ================= 3. BUILD THE DRILL-DOWN WARD-LEVEL MAP =================
import math as _math

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = _math.radians(lat1), _math.radians(lat2)
    dphi = _math.radians(lat2 - lat1)
    dlambda = _math.radians(lon2 - lon1)
    a = (_math.sin(dphi / 2) ** 2 +
         _math.cos(p1) * _math.cos(p2) * _math.sin(dlambda / 2) ** 2)
    return 2 * R * _math.asin(_math.sqrt(a))

fig = go.Figure()

# --- Layer 1: ward boundary polygons ---
fig.add_trace(go.Choroplethmapbox(
    geojson=wards_gj,
    locations=[f["properties"]["ward"] for f in wards_gj["features"]],
    z=[1] * len(wards_gj["features"]),
    featureidkey="properties.ward",
    colorscale=[[0, "rgba(255,220,0,0.08)"], [1, "rgba(255,220,0,0.08)"]],
    showscale=False,
    marker_line_width=2.2,
    marker_line_color="#1F3864",
    hovertemplate="<b>%{location}</b> Ward<extra></extra>",
    name="Wards",
))

# --- Layer 2: ward name labels at each ward's centre ---
centres = [geom.representative_point() for _, _, geom in ward_shapes]
fig.add_trace(go.Scattermapbox(
    lat=[c.y for c in centres], lon=[c.x for c in centres],
    mode="text",
    text=[w for w, _, _ in ward_shapes],
    textfont=dict(size=13, color="#1F3864"),
    hoverinfo="skip", showlegend=False, name="Ward labels",
))

# --- Layer 3: GPS points, one colour per Graduation Champion ---
palette = (px.colors.qualitative.Bold + px.colors.qualitative.Set1 +
           px.colors.qualitative.Dark24)

champions = sorted(df["champion_display"].unique())
champ_info = {}
point_trace_indices = []

for i, champ in enumerate(champions):
    grp = df[df["champion_display"] == champ]
    trace_idx = len(fig.data)
    fig.add_trace(go.Scattermapbox(
        lat=grp[LAT_COL], lon=grp[LON_COL], mode="markers",
        marker=dict(size=11, color=palette[i % len(palette)], opacity=0.9),
        name=f"{champ} ({len(grp)})",
        text=[f"<b>Champion: {champ}</b><br>"
              f"Household: {hh}<br>"
              f"Ward: {w}<br>Sub-county: {sc}<br>"
              f"Lat: {la:.5f}, Lon: {lo:.5f}"
              for hh, w, sc, la, lo in zip(grp[HH_HEAD_COL], grp["ward_final"],
                                            grp[SUBCOUNTY_COL], grp[LAT_COL], grp[LON_COL])],
        hoverinfo="text",
    ))
    point_trace_indices.append(trace_idx)
    champ_info[trace_idx] = dict(
        name=champ, n=len(grp),
        lats=grp[LAT_COL].tolist(), lons=grp[LON_COL].tolist(),
    )

# --- Layer 4 (hidden by default): per-champion path lines + distance labels ---
for trace_idx, info in list(champ_info.items()):
    i = point_trace_indices.index(trace_idx)
    lats, lons = info["lats"], info["lons"]
    seg_dists = [haversine_km(lats[k], lons[k], lats[k + 1], lons[k + 1])
                 for k in range(len(lats) - 1)]
    total_km = sum(seg_dists)

    line_idx = len(fig.data)
    fig.add_trace(go.Scattermapbox(
        lat=lats, lon=lons, mode="lines",
        line=dict(width=2.5, color=palette[i % len(palette)]),
        hoverinfo="skip", showlegend=False, visible=False,
        name=f"{info['name']} path",
    ))

    mid_lats = [(lats[k] + lats[k + 1]) / 2 for k in range(len(lats) - 1)]
    mid_lons = [(lons[k] + lons[k + 1]) / 2 for k in range(len(lons) - 1)]
    mid_idx = len(fig.data)
    fig.add_trace(go.Scattermapbox(
        lat=mid_lats, lon=mid_lons, mode="markers",
        marker=dict(size=7, color="white", opacity=0.95),
        text=[f"{d:.2f} km" for d in seg_dists],
        hoverinfo="text", showlegend=False, visible=False,
        name=f"{info['name']} distances",
    ))

    minlat, maxlat = min(lats), max(lats)
    minlon, maxlon = min(lons), max(lons)
    pad_lat = max((maxlat - minlat) * 0.25, 0.01)
    pad_lon = max((maxlon - minlon) * 0.25, 0.01)
    span_f = max(maxlat - minlat + 2 * pad_lat, maxlon - minlon + 2 * pad_lon, 0.02)
    zoom_f = max(9, min(_math.log2(360 / span_f) - 0.3, 16))

    champ_info[trace_idx].update(
        line_idx=line_idx, mid_idx=mid_idx,
        center_lat=(minlat + maxlat) / 2, center_lon=(minlon + maxlon) / 2,
        zoom=round(zoom_f, 2), total_km=round(total_km, 2),
    )

# --- Layout: auto-centre and zoom exactly on Samburu's mapped wards ---
minx, miny, maxx, maxy = unary_union([g for _, _, g in ward_shapes]).bounds
span = max(maxx - minx, maxy - miny) or 1
zoom = max(3, min(math.log2(360 / span) - 0.3, 15))
overview_center = {"lat": (miny + maxy) / 2, "lon": (minx + maxx) / 2}
overview_zoom = zoom
map_title = "Samburu County \u2013 Household GPS Points by Graduation Champion (Ward Drill-Down)"

fig.update_layout(
    mapbox_style="carto-positron",
    mapbox_zoom=overview_zoom,
    mapbox_center=overview_center,
    margin={"r": 0, "t": 60, "l": 0, "b": 0},
    height=820,
    title=dict(text=map_title, font=dict(size=17, color="#1F3864"), x=0.5),
    legend_title_text="Click a name to focus \u25b8 Graduation Champion (n households)",
    legend=dict(bgcolor="rgba(255,255,255,0.85)", bordercolor="#CCCCCC", borderwidth=1),
)

# ================= 3b. CLICK-TO-FOCUS INTERACTIVITY =================
DIV_ID = "samburu_map_div"

champ_info_js = {
    str(k): {
        "name": v["name"], "n": v["n"],
        "lineIdx": v["line_idx"], "midIdx": v["mid_idx"],
        "centerLat": v["center_lat"], "centerLon": v["center_lon"],
        "zoom": v["zoom"], "totalKm": v["total_km"],
    } for k, v in champ_info.items()
}

post_script = f"""
(function() {{
  var gd = document.getElementById('{DIV_ID}');
  var champInfo = {json.dumps(champ_info_js)};
  var pointTraceIdxs = {json.dumps(point_trace_indices)};
  var overviewCenter = {json.dumps(overview_center)};
  var overviewZoom = {overview_zoom};
  var originalTitle = {json.dumps(map_title)};
  var focusedIdx = null;

  function allLineMidIdxsAndVis(showIdx) {{
    var idxs = [];
    var vis = [];
    Object.keys(champInfo).forEach(function(k) {{
      var ci = champInfo[k];
      idxs.push(ci.lineIdx, ci.midIdx);
      var show = (parseInt(k) === showIdx);
      vis.push(show, show);
    }});
    return {{idxs: idxs, vis: vis}};
  }}

  function focusChampion(idx) {{
    var info = champInfo[idx];
    if (!info) return;

    var opac = pointTraceIdxs.map(function(pi) {{
      return (parseInt(pi) === idx) ? 1 : 0.12;
    }});
    Plotly.restyle(gd, {{opacity: opac}}, pointTraceIdxs);

    var lm = allLineMidIdxsAndVis(idx);
    Plotly.restyle(gd, {{visible: lm.vis}}, lm.idxs);

    var titleText = info.name + " \u2014 " + info.n + " household(s)" +
      (info.n > 1 ? (", path distance " + info.totalKm.toFixed(2) + " km") : "");

    Plotly.relayout(gd, {{
      'mapbox.center': {{lat: info.centerLat, lon: info.centerLon}},
      'mapbox.zoom': info.zoom,
      'title.text': titleText
    }});
  }}

  function resetView() {{
    var opac = pointTraceIdxs.map(function() {{ return 1; }});
    Plotly.restyle(gd, {{opacity: opac}}, pointTraceIdxs);

    var lm = allLineMidIdxsAndVis(-1);
    Plotly.restyle(gd, {{visible: lm.vis}}, lm.idxs);

    Plotly.relayout(gd, {{
      'mapbox.center': overviewCenter,
      'mapbox.zoom': overviewZoom,
      'title.text': originalTitle
    }});
  }}

  gd.on('plotly_legendclick', function(data) {{
    var idx = data.curveNumber;
    if (!(idx in champInfo)) return true;
    if (focusedIdx === idx) {{
      resetView();
      focusedIdx = null;
    }} else {{
      focusChampion(idx);
      focusedIdx = idx;
    }}
    return false;
  }});

  var btn = document.getElementById('reset-view-btn');
  if (btn) {{
    btn.addEventListener('click', function() {{
      resetView();
      focusedIdx = null;
    }});
  }}
}})();
"""

html_body = fig.to_html(full_html=False, include_plotlyjs="cdn", div_id=DIV_ID,
                          post_script=post_script)

full_html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>Samburu Graduation Champions Map</title>
<style>
  body {{ margin: 0; font-family: Calibri, Arial, sans-serif; }}
  #toolbar {{
    padding: 10px 16px; background: #1F3864; color: white;
    display: flex; align-items: center; justify-content: space-between;
  }}
  #toolbar span {{ font-size: 14px; }}
  #reset-view-btn {{
    background: #B08D57; color: white; border: none; padding: 8px 16px;
    border-radius: 4px; cursor: pointer; font-size: 13px; font-weight: bold;
  }}
  #reset-view-btn:hover {{ background: #9a7a4a; }}
</style>
</head>
<body>
<div id="toolbar">
  <span>Click a Graduation Champion's name in the legend to focus the map on their households and see distances between visits.</span>
  <button id="reset-view-btn">Reset View</button>
</div>
{html_body}
</body></html>
"""

with open("Samburu_Graduation_Champions_Map.html", "w", encoding="utf-8") as f:
    f.write(full_html)

# ================= 4. WARD-LEVEL SUMMARY & EXPORT =================
summary = df.groupby(["ward_final", "champion_display"]).size().unstack(fill_value=0)
print("\nHousehold points per ward per Graduation Champion:")
print(summary)

df.to_csv("Samburu_points_with_wards.csv", index=False)
summary.to_csv("Samburu_ward_champion_summary.csv")
print("\nSaved: 'Samburu_Graduation_Champions_Map.html', "
      "'Samburu_points_with_wards.csv', 'Samburu_ward_champion_summary.csv'")



472 GPS rows loaded for Samburu County.
11 unique Graduation Champions (after name cleaning).
Wards found in Samburu County boundary file: 15
444 points matched to a mapped ward boundary; 28 fell outside the boundary file (kept, using recorded ward instead).

Household points per ward per Graduation Champion:
champion_display  Cecilia lenapololo  Faith Risilah letarkush  \
ward_final                                                      
Angata Nanyokie                    0                        0   
Lodokejek                          0                       14   
Maralal                            0                        0   
Poro                               0                       60   
Suguta Marmar                      0                        0   
Wamba East                         1                        0   
Wamba North                        0                        0   

champion_display  Florence Lenkaina  Gideon Ltupusha  Hamilton Leseketeti  \
ward_final                